# PC-FNO3D réel (Grand Tunis) v4 — affinage sur les DSO (Colab, A100)

Objectif : **tous les DSO**. La v2 prolongée manque DSO1 (25 m ; R² de w), DSO2 (masse) et DSO3
(rugosité). Étapes : (1) re-mesure de la v2 (DSO5 corrigé) brute et avec **w recalculé par
continuité** ; (2) poids u, v par niveau mesurés sur la **validation** ; (3) affinage 15 époques
sélectionné sur le **score DSO** de la validation (masse ×5, gradients 0,5, w ×2) ; (4) test avec et
sans w par continuité ; (5) vraies mesures ; (6) **décision** (`scripts/decision_v4_grand_tunis.py`,
règle fixée avant le test) : la v4 n'est livrée que si elle est au moins aussi bonne que la v2 sur
DSO1, DSO4, DSO5 et les vraies mesures, sinon la v2 avec w par continuité ; (7) **DSO6** sur le modèle
retenu, sans réentraînement (ensemble par masquage, calibration sur la validation, couverture 80 % et
corrélation incertitude / erreur sur le test). Le test n'est lu qu'une fois : une étape déjà faite est
sautée. Rapport : `runs/rapport_v4.md` (DSO1 et DSO3 niveau par niveau v2 / v4, R² de w avec et sans
continuité, DSO2 « garanti par construction »). **Variantes proposées**, calculées mais jamais
utilisées pour décider tant que l'encadrant ne les a pas validées : DSO3 sur la partie prévisible de
la vérité (instants régénérés sans les perturbations aléatoires du générateur) et DSO1 sur le R² de
u et v seulement, w rapporté à part. Aucun bruit n'est ajouté aux prédictions. Seul `best.pt` va sur le Drive. La session s'arrête
à la fin. Après ça, plus aucun entraînement.

## 1. Drive, code et données

In [ ]:
from google.colab import drive
import os as _os
if not _os.path.isdir("/content/drive/MyDrive"):   # deja monte (panneau Fichiers) : ne pas redemander
    drive.mount("/content/drive")

import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

GT = Path("/content/drive/MyDrive/pcfno3d_grand_tunis")
GT.mkdir(parents=True, exist_ok=True)
JOURNAUX = GT / "journaux"
JOURNAUX.mkdir(exist_ok=True)

# code : archive deposee sur le Drive depuis ines-pc
CODE = Path("/content/PFE")
if CODE.exists():
    shutil.rmtree(CODE)
CODE.mkdir(parents=True)
with zipfile.ZipFile(GT / "pcfno3d_code.zip") as zf:
    zf.extractall(CODE)
tampon = CODE / "PCFNO3D_VERSION.txt"
print(tampon.read_text().strip() if tampon.exists() else "tampon : absent")
%cd /content/PFE
!pip install -q --no-deps -e . 2>&1 | tail -2
!pip install -q pandera xarray netCDF4 h5netcdf 2>&1 | tail -2
sys.path[:0] = ["/content/PFE", "/content/PFE/src"]

def lancer(nom: str, commande: str) -> None:
    """Commande obligatoire ; sortie affichee ET journalisee sur le Drive ; arret si echec."""
    t0 = time.time()
    with open(JOURNAUX / f"{nom}.log", "a", encoding="utf-8") as log:
        log.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} : {commande}\n")
        p = subprocess.Popen(commande, shell=True, cwd="/content/PFE", stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for ligne in p.stdout:
            print(ligne, end="")
            log.write(ligne)
            log.flush()
        p.wait()
    print(f"[{nom}] {time.time() - t0:.0f} s, code {p.returncode}")
    if p.returncode:
        raise RuntimeError(f"{nom} a echoue (code {p.returncode}) : voir {JOURNAUX / (nom + '.log')}")

# donnees : grille reelle, forcage ERA5, stations (une archive sur le Drive)
DON = Path("/content/donnees")
if not (DON / "grand_tunis" / "forcage_era5.csv").exists():
    with zipfile.ZipFile(GT / "grand_tunis_donnees.zip") as zf:
        zf.extractall(DON)
# mises a jour deposees apres coup sur le Drive (morceaux de moins de 10 Mo) :
# maj_<fichier> ou maj_<fichier>.000, .001... -> remplacent <fichier> dans donnees/grand_tunis
import collections
morceaux = collections.defaultdict(list)
for f in sorted(GT.glob("maj_*")):
    nom = f.name[4:]
    tete, _, queue = nom.rpartition(".")
    morceaux[tete if queue.isdigit() and tete else nom].append(f)
for nom, fs in morceaux.items():
    with open(DON / "grand_tunis" / nom, "wb") as out:
        for f in sorted(fs):
            out.write(f.read_bytes())
    print("mise a jour :", nom, "<-", [f.name for f in sorted(fs)])
print(sorted(p.name for p in (DON / "grand_tunis").iterdir()))

## 2. Cache (3 parties) et observations

In [ ]:
# cache : les 3 parties du Drive -> disque local ; index fusionne
LOC = Path("/content/grand_tunis")
CACHE, JEU = LOC / "cache", LOC / "jeu"
for k in (1, 2, 3):
    if not (GT / f"cache_partie_{k}.zip").exists():
        raise FileNotFoundError(f"cache_partie_{k}.zip absent : finir le carnet de generation")
if not (CACHE / "index.csv").exists():
    CACHE.mkdir(parents=True, exist_ok=True)
    for k in (1, 2, 3):
        with zipfile.ZipFile(GT / f"cache_partie_{k}.zip") as zf:
            zf.extractall(CACHE)
    with zipfile.ZipFile(GT / "jeu.zip") as zf:
        zf.extractall(JEU)
    import pandas as pd
    idx = pd.concat([pd.read_csv(CACHE / f"index_partie_{k}.csv") for k in (1, 2, 3)])
    idx.drop_duplicates("stem").to_csv(CACHE / "index.csv", index=False)
import pandas as pd
print(pd.read_csv(CACHE / "index.csv")["split"].value_counts().to_string())

# la grille complete (aneo_grid_points.csv) est dans jeu.zip
if not (GT / "best_dso_v7.pt").exists():
    raise FileNotFoundError("deposer best_dso_v7.pt (poids v7) dans MyDrive/pcfno3d_grand_tunis/")
lancer("controle", "python -u -m pcfno3d -c grand_tunis_v4_dso_colab check")

# observations reelles du domaine (affinage et jugement)
OBS = Path("/content/obs")
with zipfile.ZipFile(GT / "grand_tunis_obs.zip") as zf:
    zf.extractall(OBS)
print(sorted(p.name for p in OBS.iterdir()))

## 3. Mémoire du GPU

In [ ]:
# memoire : un pas complet (avant + arriere) sur la grille entiere ; recalcul des activations
# couche par couche si le GPU ne suffit pas (memes resultats, ~30 % plus lent)
import torch
from pcfno3d.config import Config
from pcfno3d.geometry import Geometry
from pcfno3d.model import build_model

def essai(recalcul: bool) -> float:
    cfg = Config.load("configs/grand_tunis_v4_dso_colab.yaml")
    cfg.model.recalcul_activations = recalcul
    geom = Geometry(cfg, device="cuda")
    m = build_model(cfg, geom.c_in).cuda().train()
    x = torch.randn(cfg.train.batch_size, geom.c_in, *geom.shape, device="cuda")
    torch.cuda.reset_peak_memory_stats()
    with torch.autocast("cuda", dtype=torch.bfloat16):
        m(x).float().pow(2).mean().backward()
    pic = torch.cuda.max_memory_allocated() / 1e9
    del m, x
    torch.cuda.empty_cache()
    return pic

total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(torch.cuda.get_device_name(0), f"{total:.0f} Go")
try:
    pic = essai(False)
    RECALCUL = pic > 0.85 * total
except torch.cuda.OutOfMemoryError:
    torch.cuda.empty_cache()
    pic, RECALCUL = float("nan"), True
if RECALCUL:
    pic = essai(True)
print(f"pic {pic:.1f} Go sur {total:.0f} | recalcul des activations : {RECALCUL}")
OPTS = f"--set model.recalcul_activations={str(RECALCUL).lower()}"

## 4. v2 re-mesurée, poids, affinage v4, test, vraies mesures, décision, DSO6, fin de la session

In [ ]:
from google.colab import runtime
import traceback
RUNS = GT / "runs"
V2, V4 = "grand_tunis_v2_long_colab", "grand_tunis_v4_dso_colab"
R2, R4 = RUNS / "grand_tunis_v2_long", RUNS / "grand_tunis_v4_dso"
POIDS = R4 / "uv_level_weight.json"
WCONT = "--set inference.w_continuite=true"

# l'evaluation de la v2 faite avec le code d'avant (DSO5 mal mesure) reste pour le rapport
for d in ("best", "best_wcont"):
    a = R2 / "eval_pcfno3d" / d
    if a.exists() and not a.with_name(d + "_avant_v4").exists():
        shutil.copytree(a, a.with_name(d + "_avant_v4"))

def evalue(run, dossier):                    # le test n'est calcule qu'UNE fois (code actuel :
    f = run / "eval_pcfno3d" / dossier / "test_par_niveau_synoptique.csv"   # colonne rugosite)
    return f.exists() and "rugosite" in f.read_text().splitlines()[0]

def entraine():                              # les 15 epoques de la v4 sont deja faites
    h = R4 / "history.json"
    return h.exists() and any(r.get("epoch") == 14 for r in json.loads(h.read_text()))

etapes = (
    ("evaluer_v2_brut", f"python -u -m pcfno3d -c {V2} {OPTS} evaluate", lambda: evalue(R2, "best")),
    ("evaluer_v2_wcont", f"python -u -m pcfno3d -c {V2} {OPTS} {WCONT} evaluate", lambda: evalue(R2, "best_wcont")),
    ("poids_v4", f"python -u -m pcfno3d -c {V2} {OPTS} dso-weights --out {POIDS}", POIDS.exists),
    ("entrainer_v4", f"python -u -m pcfno3d -c {V4} {OPTS} --set loss.uv_level_weight={POIDS} train", entraine),
    ("evaluer_v4_wcont", f"python -u -m pcfno3d -c {V4} {OPTS} evaluate", lambda: evalue(R4, "best_wcont")),
    ("evaluer_v4_brut", f"python -u -m pcfno3d -c {V4} {OPTS} --set inference.w_continuite=false evaluate", lambda: evalue(R4, "best")),
    ("valider_v4", f"python -u scripts/valider_grand_tunis_reel.py -c {V4} --obs {OBS}",
     lambda: (R4 / "validation_reelle" / "test" / "stations_bilan.csv").exists()),
    # variantes PROPOSEES (DSO3 sur la partie previsible de la verite) : publiees dans le
    # rapport, jamais lues par la decision tant que l'encadrant ne les a pas validees
    ("variantes_v2", f"python -u scripts/dso_variantes_grand_tunis.py -c {V2} --donnees {DON / 'grand_tunis'} --w-continuite",
     lambda: (R2 / "eval_pcfno3d" / "best_wcont" / "dso3_previsible.json").exists()),
    ("variantes_v4", f"python -u scripts/dso_variantes_grand_tunis.py -c {V4} --donnees {DON / 'grand_tunis'} --w-continuite",
     lambda: (R4 / "eval_pcfno3d" / "best_wcont" / "dso3_previsible.json").exists()),
)
try:
    for nom, cmd, fait in etapes:
        if fait():
            print(f"[{nom}] deja fait : saute")
            continue
        try:
            lancer(nom, cmd)
        except Exception:
            traceback.print_exc()

    # decision (regle d'Ines) : la v4 seulement si au moins aussi bonne que la v2 sur DSO1, DSO4,
    # DSO5 et les vraies mesures ; sinon la v2 avec w par continuite. Plus aucun entrainement.
    DEC = "python -u scripts/decision_v4_grand_tunis.py --runs " + str(RUNS)
    code_dec = subprocess.run(DEC, shell=True, cwd="/content/PFE").returncode
    if code_dec not in (0, 3):
        raise RuntimeError(f"decision impossible (code {code_dec}) : evaluations incompletes")
    conf, run, opts = (V4, R4, OPTS) if code_dec == 0 else (V2, R2, OPTS + " " + WCONT)
    print("modele retenu :", run.name, "(w par continuite)")

    # DSO6 sur le modele retenu, SANS reentrainement : ensemble par masquage, calibration sur la
    # validation, puis couverture 80 % et correlation incertitude / erreur sur le test (une fois)
    CAL = run / "fiabilite_pcfno3d" / "best_wcont" / "calibration_validation.json"
    D6 = run / "eval_pcfno3d" / "best_wcont" / "dso6_modele_livre.json"
    if not CAL.exists():
        lancer("calibrer_dso6", f"python -u -m pcfno3d -c {conf} {opts} calibrate-reliability --out {CAL}")
    if not D6.exists():
        try:
            lancer("dso6", f"python -u -m pcfno3d -c {conf} {opts} dso6 --calibration {CAL} --out {D6}")
        except RuntimeError as exc:          # code 3 = NON CONFORME : le verdict est ecrit
            print(exc)
    subprocess.run(DEC + f" --dso6 {D6}", shell=True, cwd="/content/PFE")
    print("rapport ->", RUNS / "rapport_v4.md", "| decision ->", RUNS / "decision_v4.json")
except Exception:
    traceback.print_exc()
finally:
    runtime.unassign()